# cAPTure: graph-pilot structural audit

This notebook runs only Stage 1 of the pilot: it constructs five-second snapshots in memory and audits size, connectivity, multiplicity, temporal change, benign/attack mixing, shortcuts, and resources. It does not train models and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

The recommended sequence is **SMOKE → manual review → FULL_DEV**. Results are descriptive and do not automatically approve the structural gate.

## 1. Mount Drive and prepare the environment

Before opening Colab, push these changes to the branch configured below. This section installs only the existing CPU dependencies; it does not run tests.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_graph_structural_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

required_files = [
    "code/python/requirements-capture.txt",
    "code/python/utils/capture_graph_structural.py",
    "configs/capture_experiment_v1.yaml",
    "configs/capture_packet_schema_v1.yaml",
    "configs/capture_graph_structural_audit_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display, Markdown
from utils.capture_graph_structural import run_capture_graph_structural_audit

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CPU structural-audit environment is ready.")

## 2. Bind the prepared FULL_DEV artifacts

The audit uses the already verified canonical Parquet files. It processes one scenario at a time and uses `/content` for ephemeral staging. `BATCH_SIZE` is an execution parameter, not a scientific decision; reduce it if Colab runs short of RAM.

In [ ]:
MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
STRUCTURAL_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_structural_audit_v1.yaml"
)
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
)
BATCH_SIZE = 250_000

for required_path in (
    MANIFEST_PATH, PACKET_SCHEMA_PATH, STRUCTURAL_CONFIG_PATH, PREPARED_RUN_DIR
):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")

print("Prepared input:", PREPARED_RUN_DIR)
print("Local staging:", LOCAL_WORK_ROOT)

## 3. Run SMOKE

SMOKE processes `train_empty_conn` and `train_dollar_char`, one from each benign block. To resume an existing run, replace `None` with its exact ID.

In [ ]:
RUN_SMOKE = True  # Set False only to reuse already reviewed SMOKE artifacts.
SMOKE_RUN_ID = None  # Provide the existing ID when RUN_SMOKE is False.
if not RUN_SMOKE and SMOKE_RUN_ID is None:
    raise ValueError("Provide the reviewed SMOKE_RUN_ID when RUN_SMOKE is False.")
if SMOKE_RUN_ID is None:
    SMOKE_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_structural_smoke"
    )
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "graph_pilot_runs" / SMOKE_RUN_ID
print("SMOKE run ID:", SMOKE_RUN_ID)
print("SMOKE output:", SMOKE_OUTPUT_DIR)

if RUN_SMOKE:
    SMOKE_RESULT = run_capture_graph_structural_audit(
        manifest_path=MANIFEST_PATH,
        packet_schema_path=PACKET_SCHEMA_PATH,
        structural_config_path=STRUCTURAL_CONFIG_PATH,
        prepared_run_dir=PREPARED_RUN_DIR,
        output_dir=SMOKE_OUTPUT_DIR,
        local_work_root=LOCAL_WORK_ROOT,
        mode="SMOKE",
        batch_size=BATCH_SIZE,
    )
else:
    required_smoke_outputs = [
        "scenario_summary.csv", "percentile_summary.csv",
        "resource_summary.json", "construction_summary.json",
        "shortcut_audit.json", "review.md", "run_status.json",
    ]
    missing_smoke_outputs = [
        name for name in required_smoke_outputs
        if not (SMOKE_OUTPUT_DIR / name).is_file()
    ]
    if missing_smoke_outputs:
        raise FileNotFoundError(
            f"Existing SMOKE artifacts are incomplete: {missing_smoke_outputs}"
        )
    print("Reusing previously reviewed SMOKE artifacts without rerunning them.")

## 4. Review SMOKE

Review row conservation, plausible sizes, Jaccard values, components, mixed windows, runtime, and RAM. Also open `review.md` and `shortcut_audit.json` in Drive before approving FULL_DEV.

In [ ]:
smoke_scenarios = pd.read_csv(SMOKE_OUTPUT_DIR / "scenario_summary.csv")
smoke_percentiles = pd.read_csv(SMOKE_OUTPUT_DIR / "percentile_summary.csv")
smoke_resources = json.loads(
    (SMOKE_OUTPUT_DIR / "resource_summary.json").read_text()
)
smoke_construction = json.loads(
    (SMOKE_OUTPUT_DIR / "construction_summary.json").read_text()
)

display(smoke_scenarios)
display(smoke_percentiles.loc[
    smoke_percentiles["metric"].isin([
        "nodes", "edges", "weak_components",
        "largest_weak_component_fraction", "mean_multiplicity",
        "adjacent_node_jaccard", "adjacent_pair_jaccard",
    ])
].set_index(["scenario", "metric", "percentile"]))
print("Row conservation:", smoke_construction["row_conservation_passed"])
print(json.dumps(smoke_resources, indent=2)[:12000])
display(Markdown((SMOKE_OUTPUT_DIR / "review.md").read_text()))

## 5. Manual approval for FULL_DEV

Set `APPROVE_FULL_DEV` to `True` only after reviewing SMOKE. The note remains visible in the session; document the final scientific gate in the FULL_DEV `review.md`.

In [ ]:
APPROVE_FULL_DEV = False
SMOKE_REVIEW_NOTE = ""  # Summarize observed conservation, plausibility, and resources.

if not APPROVE_FULL_DEV:
    raise RuntimeError(
        "Review the SMOKE artifacts, then set APPROVE_FULL_DEV=True."
    )
if not SMOKE_REVIEW_NOTE.strip():
    raise ValueError("Record a non-empty SMOKE_REVIEW_NOTE before FULL_DEV.")
print("FULL_DEV approved from this reviewed SMOKE:", SMOKE_RUN_ID)

## 6. Run FULL_DEV

FULL_DEV processes only the five development scenarios. Each completed scenario is immutable and verifiable; use the same `FULL_DEV_RUN_ID` to resume.

In [ ]:
FULL_DEV_RUN_ID = None  # Paste an existing ID here to resume.
if FULL_DEV_RUN_ID is None:
    FULL_DEV_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_structural_full_dev"
    )
FULL_DEV_OUTPUT_DIR = DRIVE_ROOT / "graph_pilot_runs" / FULL_DEV_RUN_ID
print("FULL_DEV run ID:", FULL_DEV_RUN_ID)
print("FULL_DEV output:", FULL_DEV_OUTPUT_DIR)

FULL_DEV_RESULT = run_capture_graph_structural_audit(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    structural_config_path=STRUCTURAL_CONFIG_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    output_dir=FULL_DEV_OUTPUT_DIR,
    local_work_root=LOCAL_WORK_ROOT,
    mode="FULL_DEV",
    batch_size=BATCH_SIZE,
)

## 7. Review FULL_DEV and complete the structural gate

Step tables are descriptive: a window containing multiple steps appears in multiple groups. Do not interpret windows as independent observations. Record the final decision in the `review.md` copy without consulting Test1 or Test2.

In [ ]:
scenario_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "scenario_summary.csv")
percentile_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "percentile_summary.csv")
step_summary = pd.read_csv(FULL_DEV_OUTPUT_DIR / "step_topology_summary.csv")
shortcut_audit = json.loads(
    (FULL_DEV_OUTPUT_DIR / "shortcut_audit.json").read_text()
)
resource_summary = json.loads(
    (FULL_DEV_OUTPUT_DIR / "resource_summary.json").read_text()
)

display(scenario_summary.set_index("scenario"))
display(percentile_summary.loc[
    percentile_summary["metric"].isin([
        "nodes", "edges", "weak_components",
        "largest_weak_component_fraction", "mean_multiplicity",
        "branching_node_fraction", "adjacent_node_jaccard",
        "adjacent_pair_jaccard",
    ])
].set_index(["scenario", "metric", "percentile"]))
display(step_summary.head(200))
print("Fold endpoint/pair memorisation diagnostics:")
print(json.dumps(shortcut_audit["fold_memorisation_rules"], indent=2)[:20000])
print("Resource summary:")
print(json.dumps(resource_summary, indent=2))
display(Markdown((FULL_DEV_OUTPUT_DIR / "review.md").read_text()))
print("Review file:", FULL_DEV_OUTPUT_DIR / "review.md")

## 8. Interpretation boundary

This audit can reveal variation, triviality, shortcuts, and resource feasibility. It does not show that a GNN uses topology. Implement the next stage only after reviewing these artifacts and freezing the memory policy, inner-checkpoint protocol, hyperparameters, and minimum effect sizes.